In [2]:
import torch
from torch import nn
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from datasets import load_dataset
from sklearn.metrics import f1_score, accuracy_score
import numpy as np

# Step 1: Load the datasets
human_dataset = load_dataset("bigcode/humanevalpack", split="test")
ai_dataset = load_dataset("Rabinovich/Code-Generation-LLM-LoRA", split="train")

# Sample sizes
human_sample_size = min(164, len(human_dataset))
ai_sample_size = min(4000, len(ai_dataset))

# Select subsets
human_dataset = human_dataset.select(range(human_sample_size))
ai_dataset = ai_dataset.select(range(ai_sample_size))

# Step 2: Preprocess datasets
tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")

def preprocess_dataset(dataset, label, column_name):
    tokenized_samples = []
    labels = []
    attention_masks = []

    for idx, code_sample in enumerate(dataset[column_name]):
        tokenized_input = tokenizer(
            code_sample,
            padding='max_length',
            truncation=True,
            max_length=512,
            return_tensors='pt'
        )
        tokenized_samples.append(tokenized_input['input_ids'].squeeze(0))
        labels.append(label)
        attention_masks.append(tokenized_input['attention_mask'].squeeze(0))

    return tokenized_samples, labels, attention_masks

# Preprocess human-written and AI-generated datasets
human_tokens, human_labels, human_attention_masks = preprocess_dataset(
    human_dataset,
    0,
    'buggy_solution'
)

ai_tokens, ai_labels, ai_attention_masks = preprocess_dataset(
    ai_dataset,
    1,
    'answer'
)

# Combine datasets
tokens = human_tokens + ai_tokens
labels = human_labels + ai_labels
attention_masks = human_attention_masks + ai_attention_masks

# Convert to PyTorch tensors
tokens = torch.stack(tokens)
labels = torch.tensor(labels)
attention_masks = torch.stack(attention_masks)

# Step 3: Define the CodeBERT model with Dropout
class CodeBERTClassifier(nn.Module):
    def __init__(self):
        super(CodeBERTClassifier, self).__init__()
        self.model = RobertaForSequenceClassification.from_pretrained("microsoft/codebert-base", num_labels=2)
        self.dropout = nn.Dropout(p=0.4)  # Add a dropout layer

    def forward(self, input_ids, attention_mask=None):
        outputs = self.model(input_ids, attention_mask=attention_mask)
        logits = self.dropout(outputs.logits)  # Apply dropout
        return logits

# Step 4: Load the trained model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CodeBERTClassifier().to(device)

# Load the model state dict
model_path = "C:/Users/ABHIMANYU/Downloads/codebert_classifier0.8525968177724407.pth"  # Update with your actual path
model.load_state_dict(torch.load(model_path))
model.eval()

# Step 5: Evaluate the model and calculate F1 score
def evaluate_model(model, tokens, labels, attention_masks):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        input_ids = tokens.to(device)
        attention_mask = attention_masks.to(device)
        labels = labels.to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        _, preds = torch.max(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return np.array(all_preds), np.array(all_labels)

# Evaluate on the validation set
predictions, true_labels = evaluate_model(model, tokens, labels, attention_masks)

# Calculate F1 score and accuracy
f1 = f1_score(true_labels, predictions, average='weighted')  # Use 'macro' or 'micro' as needed
accuracy = accuracy_score(true_labels, predictions)

print(f"F1 Score: {f1:.4f}")
print(f"Accuracy: {accuracy:.4f}")


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


OutOfMemoryError: CUDA out of memory. Tried to allocate 6.10 GiB. GPU 